# Load the same spectral image from CZI and OME-TIFF

Run in Python 3.12 with `lumic[examples]==0.1.0` installed. The first run
downloads the high-SNR COS-7 acquisition (about 145 MB) from
[BioImage Archive S-BIAD4448](https://www.ebi.ac.uk/biostudies/bioimages/studies/S-BIAD4448).
Later runs reuse the local copy.

First inspect the CZI header, then load one plane. The notebook exports that
plane to a temporary OME-TIFF and reads it back through the same Lumic API.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np

from lumic.data import download_cos7_data
from lumic.io import load_image, open_image

In [ ]:
TIME_POINT = 0
Z_SLICE = 0
CHANNEL_INDEX = 13  # Zero-based detector channel to display.

czi_path, = download_cos7_data(snr="high", progress=True)

## 1. Inspect the CZI before loading pixels

`summary()` reports the axes, shape, and available detector metadata without
decoding pixels. The channel count refers to detector channels in the file.

In [ ]:
with open_image(czi_path) as reader:
    print(reader.summary())

## 2. Load one CZI plane

`load_image()` returns a `SpectralImage`. Zero selects the first time point
and Z slice and also works when those axes are absent. To retain an entire
axis, pass `t=None` or `z=None`.

In [ ]:
czi_image = load_image(czi_path, t=TIME_POINT, z=Z_SLICE)
print(f"CZI: axes={czi_image.axes}, shape={czi_image.data.shape}, dtype={czi_image.data.dtype}")

## 3. Export one plane and inspect the OME-TIFF

Both formats use `open_image()` for inspection and `load_image()` for reading.
The temporary OME-TIFF is removed when the cell finishes. The loaded image
stays in memory for the comparison below.

In [ ]:
with TemporaryDirectory() as output_dir:
    ome_path = Path(output_dir) / "cos7-high-snr.ome.tiff"
    czi_image.save_ome_tiff(ome_path)

    with open_image(ome_path) as reader:
        print(reader.summary())

    ome_image = load_image(ome_path)

## 4. Compare pixels and detector coordinates

Lumic's OME-TIFF export preserves the pixels, dtype, axes, and available
detector coordinates for the selected plane. The check below verifies that
round trip. See the [I/O guide](../../lumic/IO_README.md) for the metadata
that Lumic preserves.

In [ ]:
assert czi_image.axes == ome_image.axes == "CYX"
assert czi_image.data.dtype == ome_image.data.dtype
np.testing.assert_array_equal(czi_image.data, ome_image.data)
np.testing.assert_array_equal(czi_image.wavelengths_nm, ome_image.wavelengths_nm)
np.testing.assert_array_equal(czi_image.detection_bands_nm, ome_image.detection_bands_nm)
print("The round trip preserved pixels, dtype, axes, and detector coordinates.")

## 5. Display one detector channel

`.data` is the ordinary NumPy array. These selected planes are CYX, so
`data[CHANNEL_INDEX]` is one YX channel image. The plotted grayscale contrast
is for display and does not change stored intensities.

In [ ]:
channel = czi_image.data[CHANNEL_INDEX]
plt.figure(figsize=(5, 5))
plt.imshow(channel, cmap="gray")
plt.title(f"CZI detector channel {CHANNEL_INDEX}")
plt.axis("off")
plt.show()

## 6. Render both spectral images as RGB

`to_rgb()` is a display conversion, not unmixing or a calibrated true-color
measurement. Reuse the first rendering's limits to compare both files with
one brightness scale and the same relative RGB channel balance.

In [ ]:
czi_rgb = czi_image.to_rgb()
ome_rgb = ome_image.to_rgb(limits=czi_rgb.limits)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(czi_rgb.data)
axes[0].set_title("Original CZI / RGB preview")
axes[1].imshow(ome_rgb.data)
axes[1].set_title("Exported OME-TIFF / RGB preview")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 7. Hand either image to the same analysis workflow

Unfolding gives the observations × detector-channels matrix used by Lumic's
numerical algorithms. It remembers how to restore the original image layout;
restoration itself does not fit or alter the data.

In [ ]:
unfolded = ome_image.unfold()  # czi_image.unfold() works identically.
restored = unfolded.fold_spectra()
np.testing.assert_array_equal(restored.data, ome_image.data)
print(f"Analysis matrix: {unfolded.data.shape}; restored image: {restored.axes}")